In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import IPython.display
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import named_arrays as na
import iris
import ctis

In [ ]:
plt.rcParams["animation.embed_limit"] = 100

In [ ]:
# scene = iris.sg.open("2013-10-22 11:30")
# scene = iris.sg.open("2014-07-05 23:00")
# scene = iris.sg.open("2014-07-04 11:40")
scene = iris.sg.open("2014-10-13 04:11") #
# scene = iris.sg.open("2014-10-07 18:16")
# scene = iris.sg.open("2014-10-08 17:01")
# obs = iris.sg.open("2015-02-24 19:03")

In [ ]:
scene = scene[{scene.axis_time: 0}]
scene.timedelta = scene.timedelta[{scene.axis_time: 0}]
scene.shape

In [ ]:
scene = na.despike(scene, axis=(scene.axis_wavelength, scene.axis_detector_y))

In [ ]:
scene.outputs = np.nan_to_num(scene.outputs)

In [ ]:
scene.outputs[scene.outputs < 0] = 0
scene.shape

In [ ]:
scene.inputs.position = scene.inputs.position - scene.inputs.position.mean()
scene.shape

In [ ]:
scene = scene.radiance
scene.shape

In [ ]:
velocity_thresh = 150 * u.km / u.s

velocity_centers = scene.velocity_doppler.cell_centers()

index_lower = np.argmax(-velocity_thresh < velocity_centers)[scene.axis_wavelength]
index_upper = np.argmax(velocity_thresh < velocity_centers)[scene.axis_wavelength]

crop_wavelength = {scene.axis_wavelength: slice(index_lower.ndarray, index_upper.ndarray)}

scene = scene[crop_wavelength]

In [ ]:
scene.shape

In [ ]:
scene.show();

In [ ]:
wavelength_rest = scene.wavelength_center

In [ ]:
AA = dict(unit=u.AA, equivalencies=u.doppler_optical(wavelength_rest))

In [ ]:
coordinates_scene = scene.inputs

In [ ]:
position_sensor = na.Cartesian2dVectorArray(
    x=na.arange(0, 512 + 1, axis="sensor_x") * u.pix,
    y=na.arange(0, 512 + 1, axis="sensor_y") * u.pix,
)

In [ ]:
coordinates_sensor = na.SpectralPositionalVectorArray(
    wavelength=scene.inputs.wavelength,
    position=position_sensor,
)

In [ ]:
angle = na.linspace(0, 360, num=4, axis="channel", endpoint=False) * u.deg

In [ ]:
channel = "dispersion angle = " + angle.to_string_array("%03d")

In [ ]:
instrument = ctis.instruments.IdealInstrument(
    area_effective=1 * u.mm ** 2,
    timedelta_exposure=10 * u.s,
    plate_scale=0.4 * u.arcsec / u.pix,
    dispersion=((5 * u.km / u.s).to(**AA) - wavelength_rest) / u.pix,
    angle=angle,
    wavelength_ref=wavelength_rest,
    position_ref=na.Cartesian2dVectorArray(256, 256) * u.pix,
    coordinates_scene=coordinates_scene,
    coordinates_sensor=coordinates_sensor,
    channel=channel,
    axis_channel="channel",
    axis_wavelength=scene.axis_wavelength,
    axis_scene_xy=(scene.axis_detector_x, scene.axis_detector_y),
    axis_sensor_xy=("sensor_x", "sensor_y"),
)

In [ ]:
%%time
images = instrument.image(scene)

In [ ]:
images = images

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(
        constrained_layout=True,
        figsize=(5, 4),
    )
    norm = plt.Normalize(
        vmin=0,
        vmax=500,
        # vmax=images.outputs.value.ndarray.max(),
    )
    colorizer = plt.Colorizer(
        cmap="gray",
        norm=norm,
    )
    ani = na.plt.pcolormovie(
        channel,
        images.inputs.position.x,
        images.inputs.position.y,
        C=images.outputs.value,
        axis_time="channel",
        ax=ax,
        kwargs_pcolormesh=dict(
            colorizer=colorizer,
        ),
    )
    plt.colorbar(
        mappable=plt.cm.ScalarMappable(colorizer=colorizer),
        ax=ax,
        label=f"signal ({images.outputs.unit:latex_inline})",
    )
    ax.set_aspect("equal")
    ax.set_xlabel(f"sensor $x$ ({images.inputs.position.x.unit})")
    ax.set_ylabel(f"sensor $y$ ({images.inputs.position.y.unit})")

result = ani.to_jshtml(fps=2)
result = IPython.display.HTML(result)

plt.close(ani._fig)

result

In [ ]:
import dataclasses

In [ ]:
coords_scene_single = coordinates_scene.replace(wavelength=images.inputs.wavelength)
coords_sensor_single = images.inputs

In [ ]:
instrument_single = dataclasses.replace(
    instrument,
    coordinates_scene=coords_scene_single,
    coordinates_sensor=coords_sensor_single,
)

In [ ]:
s = instrument_single.backproject(images) * scene.outputs.shape["wavelength"]

In [ ]:
smin = np.min(s, axis="channel")[dict(channel=0)]

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots()
    na.plt.pcolormesh(
        smin.inputs.position.x,
        smin.inputs.position.y,
        C=smin.outputs[dict(channel=0, wavelength=0)].value,
        vmin=0,
        vmax=np.percentile(smin.outputs, 99.5).value,
    )
    ax.set_aspect("equal")

In [ ]:
mart = ctis.inverters.MartInverter(
    instrument=instrument,
    # intermediate=True,
    # num_iteration=10,
)

In [ ]:
spectrum_avg = scene.outputs.mean((scene.axis_detector_x, scene.axis_detector_y))
spectrum_avg = spectrum_avg / spectrum_avg.sum()

In [ ]:
guess_spatial = smin.outputs
# guess_spatial = guess_spatial.broadcast_to(scene.outputs.shape)
guess_spatial = guess_spatial / guess_spatial.sum()
guess_spatial.sum()

In [ ]:
guess = spectrum_avg * guess_spatial * scene.outputs.sum()
# guess = spectrum_avg * scene.outputs.sum()

In [ ]:
%%time
inversion = mart(images, guess=guess, verbose=True)

In [ ]:
axis_iter = inversion.inverter.axis_iteration

In [ ]:
fig, ax = plt.subplots(
    nrows=2,
    sharex=True,
    constrained_layout=True,
)
na.plt.plot(
    inversion.iteration,
    inversion.mean_chi_squared,
    ax=ax[0],
    axis=axis_iter,
    label=channel,
)
na.plt.plot(
    inversion.iteration,
    inversion.correlation_residual,
    ax=ax[1],
    axis=axis_iter,
    label=channel,
)
ax[0].set_yscale("log")
ax[0].legend();

In [ ]:
# solution = inversion.solution[{axis_iter: ~0}]
solution = inversion.solution

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.stairs(
        scene.inputs.wavelength,
        scene.outputs.mean((scene.axis_detector_x, scene.axis_detector_y)),
        ax=ax,
        label="original",
    )
    na.plt.stairs(
        solution.inputs.wavelength,
        solution.outputs.mean((scene.axis_detector_x, scene.axis_detector_y)),
        ax=ax,
        label="reconstructed",
    )
    ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
    # ax2.set_xlabel(f"wavelength ({ax2.get_xlabel()})")
    ax.set_ylabel(f"average radiance ({ax.get_ylabel()})")
    ax.legend()

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=3,
        gridspec_kw=dict(width_ratios=[.5, .5, .1]),
        constrained_layout=True,
        figsize=(10, 6),
    )
    ax1, ax2, cax = axs
    ax2.set_yticklabels([])
    vmin = np.nanpercentile(
        a=scene.outputs,
        q=0.5,
        axis=(scene.axis_detector_x, scene.axis_detector_y),
    )
    vmax = np.nanpercentile(
        a=scene.outputs,
        q=99.5,
        axis=(scene.axis_detector_x, scene.axis_detector_y),
    )

    na.plt.rgbmesh(
        C=scene,
        axis_wavelength="wavelength",
        ax=ax1,
        vmin=vmin,
        vmax=vmax,
    )
    colorbar = na.plt.rgbmesh(
        scene.velocity_doppler,
        scene.inputs.position.x,
        scene.inputs.position.y,
        C=solution.outputs,
        axis_wavelength="wavelength",
        ax=ax2,
        vmin=vmin,
        vmax=vmax,
    )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb="wavelength",
        ax=cax,
    )
    ax1.set_title("original")
    ax2.set_title("reconstructed")
    unit_x = scene.inputs.position.x.unit
    unit_y = scene.inputs.position.y.unit
    ax1.set_xlabel(f"scene $x$ ({unit_x:latex_inline})")
    ax2.set_xlabel(f"scene $x$ ({unit_x:latex_inline})")
    ax1.set_ylabel(f"scene $y$ ({unit_y:latex_inline})")
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cax.yaxis.tick_right()
    cax.yaxis.set_label_position("right")

In [ ]:
# with astropy.visualization.quantity_support():
#     fig, ax = plt.subplots(
#         ncols=2,
#         constrained_layout=True,
#         sharex=True,
#         sharey=True,
#         # figsize=(10, 5),
#     )
#     i = {scene.axis_detector_x: 190}
#     na.plt.pcolormesh(
#         scene.inputs.wavelength,
#         scene.inputs.position.y[i],
#         C=np.sqrt(scene.outputs[i].value),
#         ax=ax[0],
#         vmin=0,
#         vmax=1000,
#     )
#     ani = na.plt.pcolormovie(
#         inversion.iteration,
#         inversion.solution.inputs.wavelength,
#         inversion.solution.inputs.position.y[i],
#         C=np.sqrt(inversion.solution.outputs[i].value),
#         ax=ax[1],
#         vmin=0,
#         vmax=1000,
#         axis_time=inversion.inverter.axis_iteration,
#     )
#     ax[0].set_title(inversion.solution.inputs.position.x[i].ndarray.mean())

# result = ani.to_jshtml(fps=10)
# result = IPython.display.HTML(result)

# ani.save("mart-iris.gif")

# plt.close(ani._fig)

# result

In [ ]:

# with astropy.visualization.quantity_support():
#     fig, axs = plt.subplots(
#         ncols=3,
#         gridspec_kw=dict(width_ratios=[.5, .5, .1]),
#         constrained_layout=True,
#         figsize=(10, 6),
#     )
#     ax1, ax2, cax = axs
#     ax2.set_yticklabels([])
#     vmax = np.nanpercentile(
#         a=scene.outputs,
#         q=99.5,
        
#         axis=(scene.axis_detector_x, scene.axis_detector_y),
#     )

#     na.plt.rgbmesh(
#         C=scene,
#         axis_wavelength="wavelength",
#         ax=ax1,
#         vmin=0,
#         vmax=vmax,
#     )
#     label = "iteration = " + inversion.iteration.to_string_array("%d")
#     chisq_str = r"$\langle \chi^2 \rangle$"
#     label = label + f"\n{chisq_str} = " + inversion.mean_chi_squared.mean("channel").to_string_array("%.03f")
#     ani, colorbar = na.plt.rgbmovie(
#         label,
#         scene.velocity_doppler,
#         scene.inputs.position.x,
#         scene.inputs.position.y,
#         C=inversion.solution.outputs,
#         axis_time=inversion.inverter.axis_iteration,
#         axis_wavelength="wavelength",
#         ax=ax2,
#         vmin=0,
#         vmax=vmax,
#     )
#     na.plt.pcolormesh(
#         C=colorbar,
#         axis_rgb="wavelength",
#         ax=cax,
#     )
#     ax1.set_title("original")
#     ax2.set_title("reconstructed")
#     unit_x = scene.inputs.position.x.unit
#     unit_y = scene.inputs.position.y.unit
#     ax1.set_xlabel(f"scene $x$ ({unit_x:latex_inline})")
#     ax2.set_xlabel(f"scene $x$ ({unit_x:latex_inline})")
#     ax1.set_ylabel(f"scene $y$ ({unit_y:latex_inline})")
#     cax.xaxis.set_ticks_position("top")
#     cax.xaxis.set_label_position("top")
#     cax.yaxis.tick_right()
#     cax.yaxis.set_label_position("right")

# result = ani.to_jshtml(fps=10)
# result = IPython.display.HTML(result)

# # ani.save("mart-iris.gif")

# plt.close(ani._fig)

# result

In [ ]:
velocity = scene.velocity_doppler

In [ ]:
sum_scene = scene.outputs.sum(scene.axis_wavelength)
sum_recon = solution.outputs.sum(scene.axis_wavelength).to(scene.outputs.unit)

In [ ]:
median_scene = na.pdf.median(velocity, scene.outputs, axis=scene.axis_wavelength)
median_recon = na.pdf.median(velocity, solution.outputs, axis=scene.axis_wavelength)

In [ ]:
iqr_scene = na.pdf.iqr(velocity, scene.outputs, axis=scene.axis_wavelength)
iqr_recon = na.pdf.iqr(velocity, solution.outputs, axis=scene.axis_wavelength)

In [ ]:
bins = dict(true=50, reconstructed=50)
hist_sum = na.histogram2d(sum_scene, sum_recon, bins=bins, min=0 * solution.outputs.unit, max=0.2e8 * solution.outputs.unit)
hist_median = na.histogram2d(median_scene, median_recon, bins=bins, min=-50 * u.km / u.s, max=50 * u.km / u.s)
hist_iqr = na.histogram2d(iqr_scene, iqr_recon, bins=bins, min=0 * u.km / u.s, max=100 * u.km / u.s)

In [ ]:
hist_sum = hist_sum / hist_sum.sum("reconstructed")
hist_median = hist_median / hist_median.sum("reconstructed")
hist_iqr = hist_iqr / hist_iqr.sum("reconstructed")

In [ ]:
import matplotlib.colors

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        constrained_layout=True,
        figsize=(10, 4),
        ncols=3,
    )
    ax_sum, ax_median, ax_iqr = axs
    na.plt.pcolormesh(
        C=hist_sum,
        ax=ax_sum,
        norm=matplotlib.colors.LogNorm(),
    )
    na.plt.pcolormesh(
        C=hist_median,
        ax=ax_median,
        norm=matplotlib.colors.LogNorm(),
    )
    na.plt.pcolormesh(
        C=hist_iqr,
        ax=ax_iqr,
        norm=matplotlib.colors.LogNorm(),
    )
    ax_sum.set_aspect("equal")
    ax_median.set_aspect("equal")
    ax_iqr.set_aspect("equal")
    ax_sum.set_title("sum")
    ax_median.set_title("median")
    ax_iqr.set_title("interquartile range")

In [ ]:
import scipy.stats

In [ ]:
where_finite = np.isfinite(sum_scene) & np.isfinite(median_scene) & np.isfinite(median_recon)
where_finite.sum() / where_finite.size

In [ ]:
float(scipy.stats.spearmanr(sum_scene[where_finite].ndarray, sum_recon[where_finite].ndarray).statistic)

In [ ]:
float(scipy.stats.spearmanr(median_scene[where_finite].ndarray, median_recon[where_finite].ndarray).statistic)

In [ ]:
float(scipy.stats.spearmanr(iqr_scene[where_finite].ndarray, iqr_recon[where_finite].ndarray).statistic)

 - signal-correlated residuals (spearman)
 - mean-chi squared of each channel
 - 

In [ ]:
%%time
predictions = instrument.image(solution, noise=False)

In [ ]:
residual = images - predictions

In [ ]:
fig, ax = na.plt.subplots(
    axis_rows="rows",
    axis_cols="cols",
    nrows=2,
    ncols=2,
    constrained_layout=True,
    figsize=(8,7),
    sharex=True,
    sharey=True,
)
ax = ax.reshape(dict(channel=-1))
vmin = -35
vmax = +35
img = na.plt.pcolormesh(
    residual.inputs.position.x,
    residual.inputs.position.y,
    C=residual.outputs.value,
    ax=ax,
    vmin=vmin,
    vmax=vmax,
    cmap="gray",
)
na.plt.set_title
na.plt.set_aspect("equal", ax=ax)
plt.colorbar(
    ax=ax.ndarray,
    mappable=img.ndarray[0],
    label=f"residual ({residual.outputs.unit:latex_inline})",
);

In [ ]:
float(scipy.stats.spearmanr(predictions.outputs.ndarray.reshape(-1), residual.outputs.ndarray.reshape(-1)).statistic)

In [ ]:
float(scipy.stats.pearsonr(predictions.outputs.ndarray.reshape(-1), residual.outputs.ndarray.reshape(-1)).statistic)

 - mulitply counts by 10 to see how it changes the results
 - smooth the outputs before taking the residual (and computing the moments)
 - negative correlation coefficient implies that we went a little too far (crossed zero)

$d \chi = \frac{d' - d}{\sqrt{d}}$ contribution of the residual to the total chi square (SNR-weighted residual)